# Mustang Startup Analysis

In [1]:
import datetime as dt
import pandas as pd
import plotly.express as px

In [2]:
# Parameters

# The fake date to use for analaysis
fake_year, fake_month, fake_day = 2024, 1, 1
fake_date: str = '-'.join([str(fake_year), str(fake_month), str(fake_day)])

# Time definition boundaries for morning and afternoon
morning_end: str = '10:00'
morning_end_timestamp: pd.Timestamp = pd.Timestamp(f'{fake_date} {morning_end}')

afternoon_start: str = '17:00'
afternoon_start_timestamp: pd.Timestamp = pd.Timestamp(f'{fake_date} {afternoon_start}')

# Set the number of minutes in each histogram bin
minutes_per_bin: int = 30

In [3]:
# Read In Google Sheet as a CSV
sheet_url: str = 'https://docs.google.com/spreadsheets/d/1LtKiSpGpBzPkVc2DjBDl_5yjR75bWKkU_Qm2IreuJik/export?format=csv'
raw_df: pd.DataFrame = pd.read_csv(sheet_url)

# Convert the times to a Pandas datetime all on the same day for analysis
# purposes and sort on the time
raw_df['time'] = pd.to_datetime(f'{fake_date} ' + raw_df['Start Time'])
raw_df['Date'] = pd.to_datetime(raw_df['Date'])
raw_df = raw_df.sort_values(by=['time'])


In [4]:
# Collect the morning events only and set the bins based on the timespan of
# events within those hours
morning_df: pd.DataFrame =raw_df[(raw_df['time'] < morning_end_timestamp)]
morning_start_hour: int = min(morning_df['time']).hour
morning_start_dt: dt.datetime = dt.datetime(fake_year, fake_month, fake_day, morning_start_hour, 0)
morning_end_hour: int = max(morning_df['time']).hour + 1
morning_end_dt: dt.datetime =  dt.datetime(fake_year, fake_month, fake_day, morning_end_hour, 0)
morning_duration: dt.timedelta = morning_end_dt - morning_start_dt
morning_bins: int = morning_duration // dt.timedelta(minutes=minutes_per_bin)

In [5]:
morning_fig = px.histogram(morning_df, x='time', nbins=morning_bins)
morning_fig.update_xaxes(type='date', 
                 tickformat='%H:%M', 
                 nticks=morning_bins, 
                 range=[morning_start_dt, morning_end_dt])
morning_fig.update_layout(bargap=0.1)
morning_fig.update_layout(
    title=f'Morning Start Times {raw_df["Date"].min().date()} to {raw_df["Date"].max().date()}',
    xaxis_title=f'Time on {minutes_per_bin} minute Bins',
    yaxis_title='Count',
)
morning_fig.show()